1. Fetch the latest 24-hour price and volume data for at least 10 popular cryptocurrencies using the Binance API and save the raw JSON response to a file named crypto_data.json.


In [1]:
import requests
import json

wanted_symbols = {
    "BTCUSDT",
    "ETHUSDT",
    "BNBUSDT",
    "XRPUSDT",
    "SOLUSDT",
    "ADAUSDT",
    "DOGEUSDT",
    "TRXUSDT",
    "LINKUSDT",
    "AVAXUSDT"
}

url = "https://api.binance.com/api/v3/ticker/24hr"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()

    all_data = response.json()


    filtered_data = [
        coin for coin in all_data
        if coin["symbol"] in wanted_symbols
    ]

    with open("crypto_data.json", "w", encoding="utf-8") as file:
        json.dump(filtered_data, file, indent=4)

    print(f"Saved {len(filtered_data)} cryptocurrencies to crypto_data.json")

except requests.exceptions.Timeout:
    print("Request timed out.")

except requests.exceptions.ConnectionError:
    print("Unable to connect to Binance.")

except requests.exceptions.HTTPError as e:
    print(f"HTTP Error: {e}")

except Exception as e:
    print(f"Error: {e}")

Saved 10 cryptocurrencies to crypto_data.json


2. Write a Python function find_most_volatile_coin(data) that takes the loaded Binance coin data and returns the symbol of the coin with the highest percentage price change in the last 24 hours.


In [2]:
def find_most_volatile_coin(data):
    """
    Returns the symbol of the coin with the highest
    absolute percentage price change in the last 24 hours.
    """

    if not data:
        return None

    most_volatile = max(
        data,
        key=lambda coin: abs(float(coin["priceChangePercent"]))
    )

    return most_volatile["symbol"]

3. Create a script that calculates the average price of all coins in your crypto_data.json, then prints a list of all coins currently trading below this average price.


In [3]:
import json

with open("crypto_data.json", "r", encoding="utf-8") as file:
    crypto_data = json.load(file)

prices = [float(coin["lastPrice"]) for coin in crypto_data]
average_price = sum(prices) / len(prices)

print(f"\nAverage Price: ${average_price:.2f}\n")

print("Coins Trading Below Average Price:")
print("-" * 35)

for coin in crypto_data:
    price = float(coin["lastPrice"])

    if price < average_price:
        print(f"{coin['symbol']}: ${price:.4f}")


Average Price: $6514.21

Coins Trading Below Average Price:
-----------------------------------
ETHUSDT: $1753.0900
BNBUSDT: $572.8300
ADAUSDT: $0.1768
XRPUSDT: $1.1395
TRXUSDT: $0.3240
LINKUSDT: $7.9150
DOGEUSDT: $0.0772
SOLUSDT: $82.6500
AVAXUSDT: $6.9260


4. Build a function rank_coins_by_volume(data) that sorts all coins by their total traded volume in descending order and prints the top 5 coins with their rank and volume.


In [4]:
def rank_coins_by_volume(data):
    """
    Sort coins by 24-hour traded volume (quoteVolume)
    in descending order and print the top 5.
    """

    sorted_coins = sorted(
        data,
        key=lambda coin: float(coin["quoteVolume"]),
        reverse=True
    )

    print("\nTop 5 Coins by Trading Volume")
    print("-" * 50)

    for rank, coin in enumerate(sorted_coins[:5], start=1):
        volume = float(coin["quoteVolume"])

        print(
            f"Rank {rank}: {coin['symbol']} | "
            f"Volume: {volume:,.2f}"
        )

5. Automate your data fetch and analysis by scheduling your script to run every hour using the schedule Python library, and add error handling to gracefully manage Binance API rate limits.<br><br><em><strong>Hint:</strong> Catch HTTP 429 errors and implement a retry with exponential backoff.</em>

In [5]:
import requests
import schedule
import time

URL = "https://api.binance.com/api/v3/ticker/24hr?symbol=BTCUSDT"

def fetch_data():
    retries = 3

    for attempt in range(retries):
        try:
            response = requests.get(URL)

            if response.status_code == 429:
                wait = 2 ** attempt
                print(f"Rate limit reached. Waiting {wait} seconds...")
                time.sleep(wait)
                continue

            response.raise_for_status()

            data = response.json()

            print("\nBitcoin Data")
            print("Price:", data["lastPrice"])
            print("24h Volume:", data["volume"])

            return

        except requests.exceptions.RequestException as e:
            print("Error:", e)

    print("Failed to fetch data after multiple attempts.")

fetch_data()

schedule.every().hour.do(fetch_data)

print("Scheduler started...")

while True:
    schedule.run_pending()
    time.sleep(1)


Bitcoin Data
Price: 62717.01000000
24h Volume: 13586.86487000
Scheduler started...


KeyboardInterrupt: 